# Drift-diffusion magnitude comparison with stimulus-dependent noise

Same comparison as [`try_ddm.ipynb`](try_ddm.ipynb), but with `FlexibleNoiseComparisonModel` instead of the scalar version: evidence noise SD is a smooth function of stimulus magnitude (B-spline of order `polynomial_order=5` by default).

- **`FlexibleNoiseComparisonModel`** — choice-only, cumulative-normal likelihood, stimulus-dependent noise.
- **`DDMFlexibleNoiseComparisonModel`** — same cognitive front-end, joint over (RT, choice) via HSSM Wiener.

Drift maps as before:
$$v = \frac{\mathrm{post}\_n_2\mu - \mathrm{post}\_n_1\mu}{\sqrt{\sigma_1(n_1)^2 + \sigma_2(n_2)^2}}$$

Note that $\sigma_1(n_1)$ and $\sigma_2(n_2)$ now depend on the stimulus, so trials with extreme magnitudes get different noise than trials with central ones. The DDM uses RT shape on top of choices; the hypothesis is that this should let us pin the noise *curve* more precisely — especially the combined `evidence_sd_combined = sqrt(σ₁² + σ₂²)` that drives drift.

Run the fit script first (~10 min for the small 4-subject smoke test, several hours for `--full`):

```bash
python notebooks/fit_flexible_ddm_garcia.py --n-subjects 4 --n-trials 200
python notebooks/fit_flexible_ddm_garcia.py --full   # all 64 subjects
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import os.path as op
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

from bauer.utils.data import load_garcia2022
from bauer.models import FlexibleNoiseComparisonModel, DDMFlexibleNoiseComparisonModel

sns.set_theme(style='ticks', context='notebook')

In [ ]:
NB_DIR = op.dirname(op.abspath(globals().get('__file__', op.join(os.getcwd(), 'placeholder'))))
if not op.isdir(NB_DIR):
    NB_DIR = '.'

for candidate in ['results_flexible_full', 'results_flexible']:
    cdir = op.join(NB_DIR, candidate)
    if op.exists(op.join(cdir, 'garcia_flex_ddm_idata.nc')):
        RESULTS_DIR = cdir
        break
else:
    RESULTS_DIR = op.join(NB_DIR, 'results_flexible')

CHOICE_PATH = op.join(RESULTS_DIR, 'garcia_flex_choice_idata.nc')
DDM_PATH = op.join(RESULTS_DIR, 'garcia_flex_ddm_idata.nc')

if not (op.exists(CHOICE_PATH) and op.exists(DDM_PATH)):
    raise FileNotFoundError(
        f'No cached fits in {RESULTS_DIR}. Run:\n'
        '  python notebooks/fit_flexible_ddm_garcia.py        # 4 subjects (small)\n'
        '  python notebooks/fit_flexible_ddm_garcia.py --full # 64 subjects'
    )

idata_choice = az.from_netcdf(CHOICE_PATH)
idata_ddm = az.from_netcdf(DDM_PATH)
print(f'loaded from {RESULTS_DIR}')
print('  choice idata:', idata_choice.posterior.sizes)
print('  DDM idata:   ', idata_ddm.posterior.sizes)

# Subset data to fit subjects
df = load_garcia2022(task='magnitude')
fit_subjects = idata_ddm.posterior.coords['subject'].values
df = df.loc[df.index.get_level_values('subject').isin(fit_subjects)].copy()
print(f'using {len(df)} trials, {len(fit_subjects)} subjects')

## Noise curves: $\sigma(n)$ across the stimulus range

The signature feature of the flexible model is that evidence noise can vary smoothly with magnitude. We rebuild a model object on the fit data so we can use `get_sd_curve` to evaluate the spline at arbitrary points, then plot the group-level posterior.

In [ ]:
# Reconstruct model objects (paradigm-aware) so we can use get_sd_curve
m_choice = FlexibleNoiseComparisonModel(paradigm=df, fit_seperate_evidence_sd=True,
                                         polynomial_order=5)
m_choice.build_estimation_model(paradigm=df, hierarchical=True)

m_ddm = DDMFlexibleNoiseComparisonModel(paradigm=df, fit_seperate_evidence_sd=True,
                                         polynomial_order=5, fit_v_scale=False)
m_ddm.build_estimation_model(paradigm=df, hierarchical=True)

x = np.linspace(df[['n1', 'n2']].min().min(), df[['n1', 'n2']].max().max(), 60)
print('evaluating at magnitudes:', x.round(1))

In [ ]:
def curve_summary(model, idata, x, variable, group=True):
    """Mean and 94% HDI of group-level noise curve at points x."""
    curve = model.get_sd_curve(idata=idata, x=x, variable=variable, group=group, hierarchical=True)
    # curve is long-format with 'x' index level; unstack to (draws, x)
    wide = curve[variable].unstack('x')
    arr = wide.values   # (n_draws, n_x)
    mean = arr.mean(axis=0)
    hdi_lo, hdi_hi = az.hdi(arr, hdi_prob=0.94).T
    return pd.DataFrame({'x': wide.columns.values, 'mean': mean,
                         'hdi_low': hdi_lo, 'hdi_high': hdi_hi})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, var in zip(axes, ['n1_evidence_sd', 'n2_evidence_sd']):
    for label, model, idata, color in [
        ('choice-only', m_choice, idata_choice, 'C0'),
        ('DDM',         m_ddm,    idata_ddm,    'C1'),
    ]:
        s = curve_summary(model, idata, x, variable=var, group=True)
        ax.fill_between(s['x'], s['hdi_low'], s['hdi_high'], alpha=0.25, color=color)
        ax.plot(s['x'], s['mean'], color=color, label=label, linewidth=2)
    ax.set(xlabel='magnitude n', ylabel='evidence_sd', title=var)
    ax.legend(fontsize=9)
fig.suptitle('Group-level noise curves with 94% HDI', y=1.02)
plt.tight_layout()

In [ ]:
def combined_curve_summary(model, idata, x):
    c1 = model.get_sd_curve(idata=idata, x=x, variable='n1_evidence_sd', group=True, hierarchical=True)
    c2 = model.get_sd_curve(idata=idata, x=x, variable='n2_evidence_sd', group=True, hierarchical=True)
    w1 = c1['n1_evidence_sd'].unstack('x')
    w2 = c2['n2_evidence_sd'].unstack('x')
    arr = np.sqrt(w1.values**2 + w2.values**2)
    mean = arr.mean(axis=0)
    hdi_lo, hdi_hi = az.hdi(arr, hdi_prob=0.94).T
    return pd.DataFrame({'x': w1.columns.values, 'mean': mean,
                         'hdi_low': hdi_lo, 'hdi_high': hdi_hi})

fig, ax = plt.subplots(figsize=(7, 4.5))
for label, model, idata, color in [
    ('choice-only', m_choice, idata_choice, 'C0'),
    ('DDM',         m_ddm,    idata_ddm,    'C1'),
]:
    s = combined_curve_summary(model, idata, x)
    ax.fill_between(s['x'], s['hdi_low'], s['hdi_high'], alpha=0.25, color=color)
    ax.plot(s['x'], s['mean'], color=color, label=label, linewidth=2)
ax.set(xlabel='magnitude n', ylabel='combined evidence_sd',
       title='sqrt(σ₁(n)² + σ₂(n)²) — the noise that drives drift')
ax.legend()
plt.tight_layout()

## HDI shrinkage of the combined noise

For each subject and each magnitude $n$, compute the per-subject combined-noise posterior, then quantify HDI tightening. We use a few representative magnitudes (the unique levels in the data).

In [ ]:
magnitudes = sorted(df['n1'].unique().tolist())
print('magnitudes:', magnitudes)

def per_subject_combined_at(model, idata, n_value):
    c1 = model.get_sd_curve(idata=idata, x=np.array([n_value], dtype=float),
                             variable='n1_evidence_sd', group=False, hierarchical=True)
    c2 = model.get_sd_curve(idata=idata, x=np.array([n_value], dtype=float),
                             variable='n2_evidence_sd', group=False, hierarchical=True)
    # Both have index (chain, draw, subject, x); pivot subject to columns
    w1 = c1['n1_evidence_sd'].unstack('subject')
    w2 = c2['n2_evidence_sd'].unstack('subject')
    arr = np.sqrt(w1.values**2 + w2.values**2)   # (n_draws, n_subjects)
    out = []
    for i, sub in enumerate(w1.columns):
        v = arr[:, i]
        lo, hi = az.hdi(v, hdi_prob=0.94)
        out.append({'subject': sub, 'mean': v.mean(), 'hdi_low': lo, 'hdi_high': hi})
    out = pd.DataFrame(out).set_index('subject')
    out['hdi_width'] = out['hdi_high'] - out['hdi_low']
    return out

rows = []
for n in magnitudes:
    s_choice = per_subject_combined_at(m_choice, idata_choice, n)
    s_ddm    = per_subject_combined_at(m_ddm,    idata_ddm,    n)
    rows.append({
        'magnitude': n,
        'median HDI width (choice)': s_choice['hdi_width'].median(),
        'median HDI width (DDM)':    s_ddm['hdi_width'].median(),
        'tighter in DDM (n/total)': f"{(s_ddm['hdi_width'] < s_choice['hdi_width']).sum()}/{len(s_ddm)}",
    })
shrinkage = pd.DataFrame(rows).set_index('magnitude')
shrinkage['ratio'] = shrinkage['median HDI width (DDM)'] / shrinkage['median HDI width (choice)']
shrinkage.round(3)

In [ ]:
# Slope plot: per-subject HDI tightening at each magnitude
fig, axes = plt.subplots(1, len(magnitudes), figsize=(2.5 * len(magnitudes), 4.5), sharey=True)
if len(magnitudes) == 1:
    axes = [axes]
for ax, n in zip(axes, magnitudes):
    s_choice = per_subject_combined_at(m_choice, idata_choice, n)
    s_ddm    = per_subject_combined_at(m_ddm,    idata_ddm,    n)
    for sub in s_choice.index:
        cw, dw = s_choice.loc[sub, 'hdi_width'], s_ddm.loc[sub, 'hdi_width']
        color = 'C2' if dw < cw else 'C3'
        ax.plot([0, 1], [cw, dw], 'o-', color=color, alpha=0.5, linewidth=1)
    ax.plot([0, 1], [s_choice['hdi_width'].mean(), s_ddm['hdi_width'].mean()],
            'ko-', linewidth=2.5, markersize=8)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['choice', 'DDM'])
    ax.set_title(f'n = {n}')
    n_tighter = (s_ddm['hdi_width'] < s_choice['hdi_width']).sum()
    n_total = len(s_ddm)
    ax.text(0.02, 0.98, f'{n_tighter}/{n_total}',
            transform=ax.transAxes, va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
axes[0].set_ylabel('combined-noise 94% HDI width')
fig.suptitle('Per-subject HDI shrinkage in combined evidence_sd, choice → DDM', y=1.02)
plt.tight_layout()

## Posterior predictive check (DDM, group means)

Same group-mean PPC structure as `try_ddm.ipynb`.

In [ ]:
ppc = m_ddm.ppc(df, idata_ddm, n_posterior_samples=200, random_seed=0,
                progressbar=False)
ppc.head()

In [ ]:
from bauer.utils.bayes import summarize_ppc_group

ppc_with_paradigm = ppc.join(df[['n1', 'n2']])
ppc_with_paradigm['log_ratio'] = np.log(ppc_with_paradigm['n2'] / ppc_with_paradigm['n1'])
ppc_with_paradigm['log_ratio_bin'] = pd.qcut(ppc_with_paradigm['log_ratio'], q=8).map(lambda x: x.mid).astype(float)
df_obs = df.copy()
df_obs['log_ratio'] = np.log(df_obs['n2'] / df_obs['n1'])
df_obs['log_ratio_bin'] = pd.qcut(df_obs['log_ratio'], q=8).map(lambda x: x.mid).astype(float)

def _wide(col):
    w = ppc_with_paradigm[col].unstack('ppc_sample')
    return w.join(ppc_with_paradigm[['log_ratio_bin']].droplevel('ppc_sample').drop_duplicates()).reset_index()

ppc_rt_wide = _wide('simulated_rt')
ppc_ch_wide = _wide('simulated_choice')

summary_rt = summarize_ppc_group(ppc_rt_wide, condition_cols=['log_ratio_bin'], hdi_prob=0.94)
summary_ch = summarize_ppc_group(ppc_ch_wide, condition_cols=['log_ratio_bin'], hdi_prob=0.94)

def obs_group_mean(df, value_col, condition_col, subject_col='subject'):
    return df.groupby([subject_col, condition_col], observed=True)[value_col].mean().groupby(condition_col, observed=True).mean()
obs_choice = obs_group_mean(df_obs, 'choice', 'log_ratio_bin').sort_index()
obs_rt = obs_group_mean(df_obs, 'rt', 'log_ratio_bin').sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
s = summary_ch.sort_index()
ax.fill_between(s.index, s['hdi025'], s['hdi975'], alpha=0.3, color='C0', label='DDM PPC 94% HDI')
ax.plot(s.index, s['p_predicted'], color='C0', label='DDM mean')
ax.plot(obs_choice.index, obs_choice.values, 'k:', marker='o', markersize=4, label='observed')
ax.set(ylabel='P(choose n2)', xlabel='log(n2/n1)', title='psychometric: PPC vs data')
ax.legend(fontsize=9)

ax = axes[1]
s = summary_rt.sort_index()
ax.fill_between(s.index, s['hdi025'], s['hdi975'], alpha=0.3, color='C0', label='DDM PPC 94% HDI')
ax.plot(s.index, s['p_predicted'], color='C0', label='DDM mean')
ax.plot(obs_rt.index, obs_rt.values, 'k:', marker='o', markersize=4, label='observed')
ax.set(ylabel='RT (s)', xlabel='log(n2/n1)', title='chronometric: PPC vs data')
ax.legend(fontsize=9)
plt.tight_layout()